# PPO Autoresearch Experiment Analysis

Analysis notebook for the PPO autoresearch setup.

This is the PPO analogue of the original LLM `analysis.ipynb`, adapted for a **single 20M-step PPO run split into 1M-step intervention segments**.

Expected files:

- `results.tsv`: high-level experiment/intervention log written by the agent.
- `runs/<run_name>/segment_metrics.tsv`: one row per 1M-step training segment.
- `runs/<run_name>/segment_metrics.jsonl`: optional richer per-segment metrics.

The primary PPO metric is not a training loss. The main quantities to inspect are:

- deterministic evaluation success rate;
- deterministic evaluation return;
- area under the success curve over environment steps;
- area under the success curve over wall-clock minutes;
- stability diagnostics such as KL, clip fraction, entropy, value loss, explained variance, SPS, and crashes.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

ROOT = Path(".")
RUNS_DIR = ROOT / "runs"

def _read_tsv(path):
    path = Path(path)
    if not path.exists():
        return None
    df = pd.read_csv(path, sep="\t")
    df.columns = [str(c).strip() for c in df.columns]
    return df

def _read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return None
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return pd.DataFrame(rows) if rows else None

def find_latest_run_dir(runs_dir=RUNS_DIR):
    runs_dir = Path(runs_dir)
    if not runs_dir.exists():
        return None
    candidates = []
    for p in runs_dir.iterdir():
        if not p.is_dir():
            continue
        has_metrics = (p / "segment_metrics.tsv").exists() or (p / "segment_metrics.jsonl").exists()
        if has_metrics:
            candidates.append(p)
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)

RUN_DIR = find_latest_run_dir()
print("Detected run dir:", RUN_DIR)

results = _read_tsv(ROOT / "results.tsv")
if results is not None:
    print(f"Loaded results.tsv: {len(results)} rows")

segment_metrics = None
if RUN_DIR is not None:
    segment_metrics = _read_tsv(RUN_DIR / "segment_metrics.tsv")
    if segment_metrics is None:
        segment_metrics = _read_jsonl(RUN_DIR / "segment_metrics.jsonl")

if segment_metrics is not None:
    print(f"Loaded segment metrics: {len(segment_metrics)} rows")
    display(segment_metrics.head(10))
else:
    print("No segment_metrics.tsv/jsonl found yet. Run at least one 1M-step segment first.")


In [ ]:
# Normalize column names and numeric types

def normalize_metrics(df):
    if df is None:
        return None
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    # Common aliases from different versions of the logger
    aliases = {
        "global_steps": "global_step",
        "step": "global_step",
        "eval_success": "eval_success_rate",
        "success_rate": "eval_success_rate",
        "final_eval_success_rate": "eval_success_rate",
        "eval_return": "eval_return_mean",
        "return_mean": "eval_return_mean",
        "final_eval_return_mean": "eval_return_mean",
        "wall_time": "wall_time_sec",
        "elapsed_sec": "wall_time_sec",
        "sps": "SPS",
    }
    for old, new in aliases.items():
        if old in df.columns and new not in df.columns:
            df[new] = df[old]

    numeric_cols = [
        "segment", "global_step", "eval_success_rate", "eval_return_mean",
        "eval_return_std", "eval_episode_len_mean", "SPS", "approx_kl",
        "clipfrac", "entropy", "value_loss", "policy_loss",
        "explained_variance", "learning_rate", "wall_time_sec",
        "segment_wall_time_sec", "peak_vram_mb", "peak_vram_gb",
        "auc_eval_success", "auc_eval_success_per_min",
        "best_eval_success_rate", "final_eval_success_rate",
    ]
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    if "global_step" not in df.columns and "segment" in df.columns:
        df["global_step"] = df["segment"] * 1_000_000

    if "wall_time_min" not in df.columns:
        if "wall_time_sec" in df.columns:
            df["wall_time_min"] = df["wall_time_sec"] / 60.0
        elif "segment_wall_time_sec" in df.columns:
            df["wall_time_min"] = df["segment_wall_time_sec"].cumsum() / 60.0

    if "peak_vram_gb" not in df.columns and "peak_vram_mb" in df.columns:
        df["peak_vram_gb"] = df["peak_vram_mb"] / 1024.0

    if "status" in df.columns:
        df["status"] = df["status"].astype(str).str.strip().str.upper()

    return df

seg = normalize_metrics(segment_metrics)
res = normalize_metrics(results)

if seg is not None:
    print("Segment metric columns:")
    print(list(seg.columns))
    display(seg.tail(10))

if res is not None:
    print("\nResults columns:")
    print(list(res.columns))
    display(res.tail(10))


## Run Summary

This section summarizes the full segmented PPO run.

Unlike the LLM notebook, the main score is not `val_bpb`. For PPO, we care about **deterministic evaluation behavior**, especially success rate and how quickly success improves.


In [ ]:
if seg is None or len(seg) == 0:
    print("No segment data available.")
else:
    s = seg.sort_values("global_step").reset_index(drop=True)

    success_col = "eval_success_rate" if "eval_success_rate" in s.columns else None
    return_col = "eval_return_mean" if "eval_return_mean" in s.columns else None

    print(f"Segments completed: {len(s)}")
    if "global_step" in s.columns:
        print(f"Last global step:   {s['global_step'].max():,.0f}")
    if "wall_time_min" in s.columns:
        print(f"Wall time:          {s['wall_time_min'].max():.1f} min")
    if "SPS" in s.columns:
        print(f"Mean SPS:           {s['SPS'].dropna().mean():,.0f}")
        print(f"Last SPS:           {s['SPS'].dropna().iloc[-1]:,.0f}" if s['SPS'].notna().any() else "Last SPS: n/a")
    if "peak_vram_gb" in s.columns:
        print(f"Peak VRAM:          {s['peak_vram_gb'].max():.1f} GB")

    if success_col:
        baseline = s[success_col].dropna().iloc[0] if s[success_col].notna().any() else np.nan
        final = s[success_col].dropna().iloc[-1] if s[success_col].notna().any() else np.nan
        best = s[success_col].max()
        best_idx = s[success_col].idxmax()
        print()
        print(f"Initial success:    {baseline:.3f}")
        print(f"Final success:      {final:.3f}")
        print(f"Best success:       {best:.3f} at segment {s.loc[best_idx, 'segment'] if 'segment' in s.columns else best_idx}")
        print(f"Success delta:      {final - baseline:+.3f}")

    if return_col:
        baseline_r = s[return_col].dropna().iloc[0] if s[return_col].notna().any() else np.nan
        final_r = s[return_col].dropna().iloc[-1] if s[return_col].notna().any() else np.nan
        best_r = s[return_col].max()
        print()
        print(f"Initial return:     {baseline_r:.3f}")
        print(f"Final return:       {final_r:.3f}")
        print(f"Best return:        {best_r:.3f}")


## Success Rate Over Training

This is the most important plot.

The x-axis is environment steps. Each point is the deterministic evaluation after a 1M-step training segment. The dashed/step line is the best success rate seen so far.


In [ ]:
if seg is None or "eval_success_rate" not in seg.columns:
    print("No eval_success_rate column found.")
else:
    s = seg.sort_values("global_step").reset_index(drop=True)
    x = s["global_step"] if "global_step" in s.columns else np.arange(len(s))
    y = s["eval_success_rate"]

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.plot(x, y, marker="o", label="Eval success rate")
    ax.step(x, y.cummax(), where="post", label="Best so far")
    ax.set_xlabel("Environment steps")
    ax.set_ylabel("Deterministic eval success rate")
    ax.set_title("PPO Autoresearch: Success Rate Across 1M-Step Segments")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()


## Success Rate Over Wall Time

This plot answers the autoresearch-style question: **how much useful policy improvement did we get per minute?**


In [ ]:
if seg is None or "eval_success_rate" not in seg.columns or "wall_time_min" not in seg.columns:
    print("Need eval_success_rate and wall_time_min columns.")
else:
    s = seg.sort_values("wall_time_min").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.plot(s["wall_time_min"], s["eval_success_rate"], marker="o", label="Eval success rate")
    ax.step(s["wall_time_min"], s["eval_success_rate"].cummax(), where="post", label="Best so far")
    ax.set_xlabel("Wall-clock minutes")
    ax.set_ylabel("Deterministic eval success rate")
    ax.set_title("PPO Autoresearch: Success Rate Per Minute")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()


## AUC Metrics

For PPO, a single final success rate can hide important differences.

Two useful aggregate scores:

- `AUC_success_vs_steps`: rewards policies that learn earlier in environment-step terms.
- `AUC_success_vs_minutes`: rewards policies that learn earlier in wall-clock terms.

The second is usually the closest PPO analogue of `val_bpb` under fixed compute pressure.


In [ ]:
def normalized_auc(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 2 or x.max() <= x.min():
        return np.nan
    order = np.argsort(x)
    x = x[order]
    y = y[order]
    return np.trapz(y, x) / (x.max() - x.min())

if seg is None or "eval_success_rate" not in seg.columns:
    print("No eval_success_rate column found.")
else:
    s = seg.copy()
    auc_steps = normalized_auc(s.get("global_step", np.arange(len(s))), s["eval_success_rate"])
    auc_min = normalized_auc(s["wall_time_min"], s["eval_success_rate"]) if "wall_time_min" in s.columns else np.nan

    print(f"AUC success vs steps:   {auc_steps:.4f}")
    print(f"AUC success vs minutes: {auc_min:.4f}")

    # Per-segment incremental AUC contribution, useful for seeing which windows mattered.
    if "global_step" in s.columns:
        s2 = s.sort_values("global_step").reset_index(drop=True)
        contrib = []
        for i in range(1, len(s2)):
            dx = s2.loc[i, "global_step"] - s2.loc[i-1, "global_step"]
            avg_y = 0.5 * (s2.loc[i, "eval_success_rate"] + s2.loc[i-1, "eval_success_rate"])
            contrib.append(dx * avg_y)
        if contrib:
            s2.loc[1:, "auc_step_contrib"] = contrib
            display(s2[["segment", "global_step", "eval_success_rate", "auc_step_contrib"]].tail(10))


## PPO Diagnostics

These plots help diagnose why a segment improved or failed.

Typical interpretations:

- `approx_kl` too high: policy updates may be too aggressive.
- `clipfrac` too high: clipping is dominating; lower LR, epochs, or clip coefficient may help.
- `entropy` collapses too early: exploration may be dying.
- `value_loss` exploding: critic instability.
- `explained_variance` increasing: critic is becoming more useful.
- `SPS` falling: intervention may be too expensive in wall-clock terms.


In [ ]:
if seg is None:
    print("No segment data available.")
else:
    s = seg.sort_values("global_step").reset_index(drop=True)
    x = s["global_step"] if "global_step" in s.columns else np.arange(len(s))

    diagnostics = [
        "approx_kl",
        "clipfrac",
        "entropy",
        "value_loss",
        "policy_loss",
        "explained_variance",
        "SPS",
        "peak_vram_gb",
    ]

    for col in diagnostics:
        if col not in s.columns or s[col].dropna().empty:
            continue
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.plot(x, s[col], marker="o")
        ax.set_xlabel("Environment steps")
        ax.set_ylabel(col)
        ax.set_title(col)
        ax.grid(True, alpha=0.3)
        plt.show()


## Intervention Log

This table connects code changes to subsequent PPO behavior.

Because the agent edits `train.py` after each 1M-step segment, each row should describe the intervention that affected the **next** segment, or the intervention just completed depending on how your logger writes it.


In [ ]:
if seg is None:
    print("No segment data available.")
else:
    cols = [
        "segment", "global_step", "commit", "status", "intervention",
        "description", "eval_success_rate", "eval_return_mean",
        "approx_kl", "clipfrac", "entropy", "value_loss", "SPS",
    ]
    cols = [c for c in cols if c in seg.columns]
    if cols:
        display(seg.sort_values("global_step")[cols].tail(25))
    else:
        display(seg.tail(25))


## Segment-to-Segment Deltas

This section measures whether the previous intervention was followed by improvement or degradation.


In [ ]:
if seg is None or "eval_success_rate" not in seg.columns:
    print("Need eval_success_rate.")
else:
    s = seg.sort_values("global_step").reset_index(drop=True).copy()
    s["delta_success"] = s["eval_success_rate"].diff()
    if "eval_return_mean" in s.columns:
        s["delta_return"] = s["eval_return_mean"].diff()
    if "SPS" in s.columns:
        s["delta_SPS"] = s["SPS"].diff()

    cols = ["segment", "global_step", "commit", "intervention", "description", "eval_success_rate", "delta_success"]
    if "eval_return_mean" in s.columns:
        cols += ["eval_return_mean", "delta_return"]
    if "SPS" in s.columns:
        cols += ["SPS", "delta_SPS"]
    cols = [c for c in cols if c in s.columns]

    display(s[cols].tail(25))

    best_delta = s["delta_success"].idxmax()
    worst_delta = s["delta_success"].idxmin()
    print("Best segment-to-segment success jump:")
    display(s.loc[[best_delta], cols])
    print("Worst segment-to-segment success drop:")
    display(s.loc[[worst_delta], cols])


## Results TSV Summary

If `results.tsv` exists, this section summarizes agent decisions: `KEEP`, `DISCARD`, `CRASH`, or `QUARANTINE`.

For segmented PPO, `results.tsv` is usually a higher-level research log, while `segment_metrics.tsv` is the detailed training trace.


In [ ]:
if res is None:
    print("No results.tsv found.")
else:
    if "status" in res.columns:
        counts = res["status"].value_counts(dropna=False)
        print("Agent decisions:")
        print(counts.to_string())

        n_keep = counts.get("KEEP", 0)
        n_discard = counts.get("DISCARD", 0)
        n_crash = counts.get("CRASH", 0)
        n_quarantine = counts.get("QUARANTINE", 0)
        n_decided = n_keep + n_discard + n_crash + n_quarantine
        if n_decided:
            print()
            print(f"Keep rate among all logged decisions: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

    display(res.tail(25))


## Best Interventions

Ranks interventions by immediate improvement in evaluation success after a segment.

This is not a perfect causal analysis, because PPO is noisy and interventions accumulate, but it is a useful first pass.


In [ ]:
if seg is None or "eval_success_rate" not in seg.columns:
    print("Need segment metrics with eval_success_rate.")
else:
    s = seg.sort_values("global_step").reset_index(drop=True).copy()
    s["delta_success"] = s["eval_success_rate"].diff()
    hits = s.dropna(subset=["delta_success"]).sort_values("delta_success", ascending=False)

    cols = ["segment", "global_step", "delta_success", "eval_success_rate", "commit", "intervention", "description"]
    cols = [c for c in cols if c in hits.columns]
    display(hits[cols].head(15))

    if len(hits):
        print(f"Total net success change: {s['eval_success_rate'].iloc[-1] - s['eval_success_rate'].iloc[0]:+.3f}")
        print(f"Sum of positive deltas:   {hits.loc[hits['delta_success'] > 0, 'delta_success'].sum():+.3f}")
        print(f"Sum of negative deltas:   {hits.loc[hits['delta_success'] < 0, 'delta_success'].sum():+.3f}")


## Final Recommendation Notes

Use this final cell to write human-readable conclusions after inspecting the plots.

Suggested questions:

1. Did success improve steadily or only after a specific intervention?
2. Is the best policy also the final policy, or did the run regress?
3. Did KL/clipfrac indicate overly aggressive updates?
4. Did entropy collapse before success appeared?
5. Did any intervention improve success but reduce SPS too much?
6. Which interventions deserve a clean confirmation run?


In [ ]:
# Fill this in manually after running the notebook.
notes = {
    "best_intervention": "",
    "likely_failure_modes": [],
    "next_experiments": [],
    "confirmation_runs": [],
}
notes
